In [1]:
# ---- Cell 1 code ----
# NOTE: the debug log showed the file is 'raw_hr_data.xlsx.csv' (a CSV with a double extension), so Excel patterns miss it
import os                                                                                     # Builds Windows-safe file paths
import glob                                                                                   # Finds files by extension pattern
import re                                                                                     # Cleans special characters in headers
import pandas as pd                                                                           # Core table loading and cleaning
import numpy as np                                                                            # Numeric helpers for dtype checks

INPUT_DIR   = 'C:/Users/majid/Desktop/Data Analyst Portfolio/2_HR_Attrition_Analytics/Data/'  # Confirmed by the debug cell
OUTPUT_PATH = os.path.join(INPUT_DIR, 'cleaned_hr_attrition_master.csv')                      # Export target sits beside the raw file

source_files = []                                                                             # Collector for every raw file found
for pattern in ['*.xlsx', '*.xls', '*.csv']:                                                  # CSV added because the raw file is really a CSV
    source_files.extend(glob.glob(os.path.join(INPUT_DIR, pattern)))                          # Avoids hardcoded file names

source_files = sorted(
    f for f in source_files
    if not os.path.basename(f).startswith(('~$', 'cleaned_'))                                 # Skip Excel lock files and our own export so re-runs never reload it
)

assert os.path.isdir(INPUT_DIR), f"Input folder not found: {INPUT_DIR}"                       # Catches a wrong path before anything else runs
assert len(source_files) > 0, "No raw files found in the input folder"                        # Fails early with a clear message

print(f"--- Raw files found in Data/ | count: {len(source_files)} ---")                       # Console divider for readable logs
for i, f in enumerate(source_files, start=1):                                                 # Numbered listing makes duplicates obvious
    print(f"  {i}. {os.path.basename(f)}")

if len(source_files) > 1:                                                                     # Project expects exactly one raw file
    print("WARNING: multiple raw files found, using the first one alphabetically")

source_file = source_files[0]                                                                 # Fixed pick keeps the run reproducible
print(f"--- Source file selected: {os.path.basename(source_file)} ---")

--- Raw files found in Data/ | count: 1 ---
  1. raw_hr_data.xlsx.csv
--- Source file selected: raw_hr_data.xlsx.csv ---


In [2]:
# ---- Cell 2 code ----
def load_raw(path):
    if path.lower().endswith('.csv'):
        try:
            return pd.read_csv(path, encoding='utf-8-sig')                     # utf-8-sig strips hidden BOM characters
        except UnicodeDecodeError:
            return pd.read_csv(path, encoding='latin-1')                       # Fallback for legacy Windows encodings
    engine = 'xlrd' if path.lower().endswith('.xls') else 'openpyxl'
    return pd.read_excel(path, engine=engine)                                  # True Excel workbooks

raw_df = load_raw(source_file)                                                 # Load once and never modify this frame
print(f"--- raw_df | shape: {raw_df.shape} ---")                                # Baseline size for later assertions

def clean_hr_table(df):
    df = df.copy()                                                             # Insulates raw_df from structural updates
    
    # Standardize column naming logic to lowercase snake_case structures
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
    df.columns = [re.sub(r'[^a-z0-9_]+', '_', c).strip('_') for c in df.columns]
    assert df.columns.is_unique, f"Duplicate headers after cleaning: {list(df.columns)}"
    
    # Dynamic identity index row detection
    id_col = next((c for c in df.columns if c in ['employeenumber', 'employee_number', 'employeeid', 'employee_id', 'emp_id', 'empid']), None)
    assert id_col is not None, f"No employee ID column found in: {list(df.columns)}"
    
    df[id_col] = df[id_col].astype(str).str.strip().str.replace(r'\.0$', '', regex=True) # Strip accidental trailing float decimals
    df[id_col] = pd.to_numeric(df[id_col], errors='coerce')                    # Bad unparseable lines turn into NaN keys
    df = df.dropna(subset=[id_col])                                            # Purge missing primary index rows
    df[id_col] = df[id_col].astype(int)                                        # Force a solid clean integer key mapping
    df = df.drop_duplicates(subset=[id_col], keep='first')                     # Enforce unique employee constraints
    
    # Separate numeric and categorical arrays to apply proper imputation rules
    num_cols = df.select_dtypes(include='number').columns
    txt_cols = df.select_dtypes(exclude=['number', 'datetime', 'bool']).columns
    df[num_cols] = df[num_cols].fillna(0)                                      # Fill missing integers with safe 0
    df[txt_cols] = df[txt_cols].fillna('Not Specified')                        # Fill missing descriptive blocks with standard labels
    
    # Programmatically identify and drop zero-variance columns to optimize warehouse memory
    known_constants = ['employeecount', 'employee_count', 'over18', 'over_18', 'standardhours', 'standard_hours']
    auto_constants = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
    drop_cols = sorted(set(auto_constants) | {c for c in known_constants if c in df.columns})
    df = df.drop(columns=drop_cols)
    print(f"--- Dropped zero-variance columns: {drop_cols} ---")
    
    return df.reset_index(drop=True), id_col                                   # Reset primary record rows for alignment tests

hr_clean, id_col = clean_hr_table(raw_df)                                      # Run data processing script

# Operational matrix quality controls
assert len(hr_clean) <= len(raw_df), "Processing error: record rows grew incorrectly"
assert hr_clean[id_col].is_unique, "Quality Control Failure: Non-unique employee identifier index matches found"
assert pd.api.types.is_integer_dtype(hr_clean[id_col]), "Formatting error: primary tracking key failed integer casting"
assert hr_clean.select_dtypes(exclude='datetime').isna().sum().sum() == 0, "Warehouse check failed: unhandled NULL parameters found"
assert all(c == c.lower() and ' ' not in c for c in hr_clean.columns), "Database constraint failure: invalid header styles found"
assert all(hr_clean[c].nunique() > 1 for c in hr_clean.columns), "Optimization failure: unpurged constant column remains"

hr_clean.to_csv(OUTPUT_PATH, index=False)                                      # Stream clean matrix back into local folder

print("\n--- Clean HR data | shape: {hr_clean.shape} ---")
print(f"Raw rows: {len(raw_df):,} | Clean rows: {len(hr_clean):,} | Removed: {len(raw_df) - len(hr_clean):,}")
print(f"Updated headers: {list(hr_clean.columns)}")
print(f"Saved to: {OUTPUT_PATH}")


--- raw_df | shape: (1470, 35) ---
--- Dropped zero-variance columns: ['employeecount', 'over18', 'standardhours'] ---

--- Clean HR data | shape: {hr_clean.shape} ---
Raw rows: 1,470 | Clean rows: 1,470 | Removed: 0
Updated headers: ['age', 'attrition', 'businesstravel', 'dailyrate', 'department', 'distancefromhome', 'education', 'educationfield', 'employeenumber', 'environmentsatisfaction', 'gender', 'hourlyrate', 'jobinvolvement', 'joblevel', 'jobrole', 'jobsatisfaction', 'maritalstatus', 'monthlyincome', 'monthlyrate', 'numcompaniesworked', 'overtime', 'percentsalaryhike', 'performancerating', 'relationshipsatisfaction', 'stockoptionlevel', 'totalworkingyears', 'trainingtimeslastyear', 'worklifebalance', 'yearsatcompany', 'yearsincurrentrole', 'yearssincelastpromotion', 'yearswithcurrmanager']
Saved to: C:/Users/majid/Desktop/Data Analyst Portfolio/2_HR_Attrition_Analytics/Data/cleaned_hr_attrition_master.csv
